In [1]:
import sys
import argparse
import concurrent.futures
import glob
import logging
import os
import time
import traceback
import multiprocessing as mp

from multiprocessing import Process
from multiprocessing import Queue

# set environment vars before numpy import
from marsdataio.os.setenv import set_env_vars, set_all_random_seed

set_env_vars(1)

import cv2

cv2.setNumThreads(0)

import numpy as np
import torch

from visdom import Visdom

from marsdataio.logging import init_logger, deinit_logger
from marsdataio.os.multiprocess import NoDaemonPool
from dataengine.generator.drivemode.drivemode import DriveModeGeneratorHandler
from dataengine.generator.renderer import RenderHandler
from dataengine.generator.nav.nav import NavGenerator, plot_legs_map

# from dataengine.generator.obstacle.segment_tracker import CameraTrackingHandler
from dataengine.generator.obstacle.obstacle import ObstacleGeneratorHandler
from dataengine.generator.obstacle.radar_assoc import ObstacleRadarAssocHandler
from dataengine.generator.road.road import RoadDataGeneratorHandler
from marsdataio.dbhelper import (
    CameraTopic,
    DbReader,
    extract_cam_timestamps,
    get_all_radar_configs,
    get_db_config,
    get_radar_topic,
    run_segmented_databases,
    trim_db_time,
)
from marsdataio.npyhelper import (
    get_all_cameras_params,
    get_db_and_video_paths,
    extract_nav_leg_graph,
    extract_npy,
)
from marstransform.geotrans import ecef2geodetic
from marsneuralzoo.models.drivenet import DriveNet
from marsneuralzoo.models.segnet import Segnet
from marsneuralzoo.models.yolo import Yolo
from marsneuralzoo.models.yolo_seg import SegmentationYolo
from marsneuralzoo.models.e2emvm import E2emvm


# 전달할 인자를 직접 설정
sys.argv = [
    '--n_process=1',
    '--src_base_dir',
    '/media/vol/shared/mars_dataset/ego_motion/2023_10/',
    '--output_dir',
    '/home/mars/mars_test/',
    '--npy_prefixes',
    # '2023_10_12_11_13_43_seg_8',
    # '2023_02_27_15_21_49_seg_175_seq_3',
    # '2023_11_01_07_30_47_seg_241',
    '2023_10_19_07_21_48_seg_53',
    '--s',
    '200',
    '--u',
    '1',
    '--override_output',
    '--debug',
]

# argparse 설정
parser = argparse.ArgumentParser()

parser.add_argument(
    '--vis',
    required=False,
    help='enable visualisation',
    dest='vis',
    action='store_true',
)
parser.add_argument(
    '--npy_prefixes',
    nargs='+',
    required=False,
    default=[],
    help='npy name prefixes, can be more than one, e.g., 2020_09_07 2020_09_07',
)
parser.add_argument(
    '--npy_list_file',
    required=False,
    help=(
        'text file containing list of npy files to process\n'
        'each line should contain the path to an npy file relative to '
        '--src_base_dir'
    ),
)
parser.add_argument(
    '--src_base_dir',
    required=False,
    help='source base dir',
    default='/media/vol/runs/ego_motion',
)
parser.add_argument(
    '--output_dir',
    required=False,
    help='output base path',
    default='/media/vol/runs/unified_data',
)
parser.add_argument(
    '--n_process',
    required=False,
    help='number of processes',
    default=9,
    type=int,
)
parser.add_argument(
    '--s',
    required=False,
    help='npy start time in second',
    default=0,
    type=int,
)
parser.add_argument(
    '--u',
    required=False,
    help='npy duration in second',
    default=0,
    type=int,
)
parser.add_argument(
    '--no_obstacle',
    required=False,
    help='exclude obstacles from data generation',
    dest='no_obstacle',
    action='store_true',
)
parser.add_argument(
    '--no_laneline',
    required=False,
    help='exclude laneline from data generation',
    dest='no_laneline',
    action='store_true',
)
parser.add_argument(
    '--no_drivemode',
    required=False,
    help='exclude drive mode from data generation',
    dest='no_drivemode',
    action='store_true',
)
parser.add_argument(
    '--no_nav',
    required=False,
    help='exclude navigation from data generation',
    dest='no_nav',
    action='store_true',
)
parser.add_argument(
    '--override_output',
    required=False,
    help='override the npy if exist, else skip the datagen if the npy exists',
    dest='override_output',
    action='store_true',
)
parser.add_argument(
    '--debug',
    required=False,
    help='set this flag to enable debug output',
    dest='debug',
    action='store_true',
)
parser.set_defaults(vis=False, override_output=False, debug=False)

# 인자 파싱
options = parser.parse_args()

configs = {
    'db_base_path': '/media/vol/shared/db?',
    'src_base_path': options.src_base_dir,
    'output_base_path': options.output_dir,
    'diskcache_dir': '/tmp/diskcache',
    'visualisation': options.vis,
    'remote_vis': False,
    'x_range': 200,
    'lane_model_path': '/media/vol/shared/runs/model/drivenet/drivenet_v100_286_best.pt',
    'seg_model_path': '/media/vol/shared/runs/model/segnet/comma10k/segnet_v112_60.pt',
    'obj_det_model_path': '/media/vol/shared/runs/detection_run/model/yolov3/marsdeepdrive/yolov3_v100_100_last_608.pt',
    'ego_mask_dir': '/media/vol/shared/obstacle/ego_mask/latest',
    'cuboid_prior_wlh': np.array([2.0, 4.0, 1.5]),
    'path_width': 4.0,
    'n_subprocesses': 4,
    'debug': options.debug,
    'gen_obstacle': not options.no_obstacle,
    'gen_laneline': not options.no_laneline,
    'gen_drivemode': not options.no_drivemode,
    'gen_nav': not options.no_nav,
    'osm_psql_params': {
        'dbname': 'osm',
        'user': 'osmuser',
        'password': 'osmuser',
        'host': '1.deep.local.marsauto.io',
        'port': '5432',
    },
    'valhalla_url': 'http://1.deep.local.marsauto.io:8002',
    'start_time': options.s,
    'duration': options.u,
}
# mp.set_start_method('spawn')
session = time.strftime('%Y-%m-%d_%H_%M_%S')
error_log_path = os.path.join(
    configs['output_base_path'], f'error_{session}.log'
)

# Init error logger to be used in child processes.
# logger_queue = init_logger(
#     stdout_lvl=None,
#     file_cfg=(logging.ERROR, error_log_path),
#     enqueue=True,
# )
# Init stdout logger
init_logger(reset_logger=False)

npy_files = []
for npy_prefix in options.npy_prefixes:
    # search inside the `YYYY-MM` directory
    str_month = f'{npy_prefix[:7]}'
    npy_files.extend(
        glob.glob(f'{options.src_base_dir}/{str_month}*/{npy_prefix}*.npy')
    )
    # search inside the `src_base_dir` directory
    npy_files.extend(glob.glob(f'{options.src_base_dir}/{npy_prefix}*.npy'))

if options.npy_list_file:
    with open(options.npy_list_file, 'r') as f:
        npy_list = f.readlines()
        # ignore lines that start with `#`
        npy_list = [npy.strip() for npy in npy_list if npy[0] != '#']
        npy_list = [f'{options.src_base_dir}/{npy}' for npy in npy_list]
        npy_files.extend(npy_list)

# npy_name -> npy_path
npy_files = {os.path.split(npy_file)[1]: npy_file for npy_file in npy_files}

if not options.override_output:
    exists_files = {}
    for npy_name, npy_file in npy_files.items():
        if os.path.exists(f'{options.output_dir}/{npy_name}'):
            exists_files[npy_name] = npy_file

    npy_names = npy_files.keys() - exists_files.keys()
    npy_files = {npy_name: npy_files[npy_name] for npy_name in npy_names}
    if exists_files:
        logging.warning(
            f'Skip {len(exists_files)} npy(s) (already generated), use the'
            f' `--override_output` option to regenerate again.'
        )


npy_files = list(npy_files.values())
if len(npy_files) == 0:
    logging.error(
        f'No matching npy is found for npy_prefixes={options.npy_prefixes} '
        f'or files specified in npy_list_file={options.npy_list_file}'
    )

args = [(configs, npy_file) for npy_file in npy_files]
configs, npy_path = args[0]

set_all_random_seed(0)
np.set_printoptions(precision=3, suppress=True)

vis = Visdom() if configs['remote_vis'] else None
out_dir_path = configs['output_base_path']
npy_filename = os.path.basename(npy_path)
npy_name = os.path.splitext(npy_filename)[0]
out_log_path = os.path.join(out_dir_path, f'{npy_name}.log')

# init_logger(
#     stdout_lvl=logging.INFO,
#     file_cfg=(logging.DEBUG, out_log_path),
#     logger_queue=logger_queue,
# )

npy_data = np.load(npy_path)

db_paths, _ = get_db_and_video_paths(npy_data, configs['db_base_path'])
db = DbReader(db_paths[0])
tname = str(npy_data['vehicle_info']['tname'])

s_time = configs['start_time']
duration = configs['duration']

sensors = npy_data['sensors']
cams_params = get_all_cameras_params(sensors)
main_cam_topic = CameraTopic.search_primary_driving_topic(cams_params.keys())
main_cam_params = cams_params[main_cam_topic]
db_configs = get_db_config(db)
radar_cfgs = get_all_radar_configs(db_configs, tname)
main_radar_topic = get_radar_topic(db)
device = torch.device('cuda')

try:
    db_s_time, duration = trim_db_time(
        db, s_time, duration, npy_data, look_ahead_dist=0
    )
except ValueError as e:
    # logging.error(f'skip {npy_name} due to {e}')
    exit()


obj_seg_model = SegmentationYolo(device=device)
e2emvm = E2emvm(multiview=False)

kj/filesystem-disk-unix.c++:1690: warning: PWD environment variable doesn't match current directory; pwd = /home/mars


Loaded SuperPoint model


In [2]:
import logging
import traceback
import os
import cv2
import numpy as np
import torch
import networkx as nx
import itertools
from itertools import count
from typing import Dict, List, Optional
from lapsolver import solve_dense
from scipy.sparse import csr_matrix
from sknetwork.clustering import Leiden

from dataengine.generator.obstacle.debug_utils import draw_mask, Statistics
from dataengine.generator.obstacle.assignment import assign_segments
from dataengine.generator.obstacle.mask import (
    load_ego_masks,
    remove_ego_body_from_masks,
    remove_overlapped_area,
)

from marsneuralzoo.models.yolo_seg import SegmentationYolo
from marsneuralzoo.models.e2emvm import E2emvm
from marsdataio.dbhelper import (
    extract_image,
    collate_images,
    DbTopic,
    DbHandler,
)
from marsdataio.npyhelper import get_all_cameras_params
from marsdataio.videowriter import VideoWriter
from marsdataio.npyrenderer.renderer import generate_colors
from collections import defaultdict
Statistics.clear()
from dataengine.generator.obstacle.assignment import assign_segments

np.set_printoptions(precision=3, suppress=True)


class SegmentTracklet:
    """
    Track information of segment. Contains it's unique id and measurements from
    camera. Uses camera id (timestamp and index) as a key and mask from camera
    will be a value. Predicted mask should be updated when track is missed.
    """

    id_counter = count()

    def __init__(self):
        self.id = next(self.id_counter)
        # cam_to_segment_mask[timestamp][cam_index] -> mask
        self.cam_to_segment_mask: Dict[int, Dict[int, np.ndarray]] = {}
        self.missed_count = 0
        self.predicted_mask: np.ndarray = None
        self.merged = False

    def add_cam_segment_mask(
        self, timestamp: int, cam_index: int, segment_mask: np.ndarray
    ):
        """Add new segment mask detected from camera.
        :param timestamp: timestamp
        :param cam_index: cam_index
        :param segment_mask: segment mask
        """
        self.cam_to_segment_mask.setdefault(timestamp, {})[
            cam_index
        ] = segment_mask

        self.missed_count = 0
        self.predicted_mask = None

    def update_predicted_mask(self, mask):
        """
        When tracking fails, use the predicted mask to track in the next frame
        """
        self.predicted_mask = mask

    def query_segment_mask(self, timestamp: int, cam_index: int):
        """query segment mask detected from camera.
        :param timestamp: timestamp
        :param cam_index: cam_index
        """
        if timestamp not in self.cam_to_segment_mask:
            raise KeyError(
                f'Timestamp {timestamp} not found in cam_to_segment_mask'
            )

        if cam_index not in self.cam_to_segment_mask[timestamp]:
            raise KeyError(
                f'Cam index {cam_index} not found for timestamp {timestamp}'
            )

        return self.cam_to_segment_mask[timestamp][cam_index]

    def track_count(self):
        return len(self.cam_to_segment_mask)




def dfs(graph, node, visited, component):
    visited.add(node)
    component.add(node)
    for neighbor in graph[node].keys():  # 2차원 구조에서 인접 노드 가져오기
        if neighbor not in visited:
            dfs(graph, neighbor, visited, component)


class SegmentTracker:
    """
    This class tracks the segments detected by segment model using optical flow.
    """

    def __init__(
        self,
        cam_index: int,
        id_tracklet_dict: Dict[int, SegmentTracklet],
        max_missed_track=12,
        min_iou_thresh=0.3,
    ):
        """Initialises states with tracklet_database."""
        self.cam_index = cam_index
        self.prev_timestamp = -1
        self.prev_gray: Optional[np.ndarray] = None  # H x W

        # id_tracklet_dict[id] -> tracklet
        self.id_tracklet_dict = id_tracklet_dict

        # missed_tracklets[id] -> tracklet
        self.missed_tracklets: Dict[int, SegmentTracklet] = {}

        # tracked_tracklets[id] -> tracklet
        self.tracked_tracklets: Dict[int, SegmentTracklet] = {}
        self.grid = None

        self.max_missed_track = max_missed_track
        self.min_iou_thresh = min_iou_thresh

    def track(self, timestamp, image, segs):
        """Takes a new camera measurements and tracks currently activated
        segment tracklets.
        :param timestamp: timestamp
        :param image: image
        :param segs: segment masks from deep model
        :return: recent tracklet ids.
        """

        h, w = image.shape[:2]
        shape = image.shape[:2]

        Statistics.startTimer('SegmentTracker_flow')
        curr_gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        curr_gray = cv2.resize(curr_gray, (w // 2, h // 2))

        if self.prev_gray is None:
            self.prev_timestamp = timestamp
            self.prev_gray = curr_gray

            h, w = image.shape[:2]
            remap = np.meshgrid(
                np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32)
            )
            self.grid = np.stack(remap, axis=2)

        flow = self._calculate_flow(self.prev_gray, curr_gray)
        flow = cv2.resize(flow, (w, h)) * 2
        Statistics.stopTimer('SegmentTracker_flow')

        warping_matrix = self.grid - flow

        Statistics.startTimer('SegmentTracker_remap')

        # project active tracklets' ids into image
        tracked_id_image = np.full(shape, -1.0, dtype=np.float32)
        missed_id_image = np.full(shape, -1.0, dtype=np.float32)

        for id, seg_trl in self.tracked_tracklets.items():
            seg = seg_trl.query_segment_mask(
                self.prev_timestamp, self.cam_index
            )
            tracked_id_image[seg > 0] = id

        for id, seg_trl in self.missed_tracklets.items():
            seg = seg_trl.predicted_mask
            missed_id_image[seg > 0] = id

        tracked_id_image = cv2.remap(
            tracked_id_image,
            warping_matrix,
            None,
            interpolation=cv2.INTER_NEAREST,
            borderMode=cv2.BORDER_REFLECT,
        )

        missed_id_image = cv2.remap(
            missed_id_image,
            warping_matrix,
            None,
            interpolation=cv2.INTER_NEAREST,
            borderMode=cv2.BORDER_REFLECT,
        )

        Statistics.stopTimer('SegmentTracker_remap')

        tracked_id_image = tracked_id_image.astype(int)
        tracked_ids = np.unique(tracked_id_image.flatten())
        tracked_ids = tracked_ids[tracked_ids > -1]
        tracked_ids = list(tracked_ids)

        missed_id_image = missed_id_image.astype(int)
        missed_ids = np.unique(missed_id_image.flatten())
        missed_ids = missed_ids[missed_ids > -1]
        missed_ids = list(missed_ids)

        prev_ids = tracked_ids + missed_ids

        Statistics.startTimer('SegmentTracker_assign')
        Statistics.startTimer('SegmentTracker_assign_0')

        segs_len = len(prev_ids)
        expected_segs = np.zeros((segs_len, h, w), dtype=np.uint8)
        missed_offset = 0
        for b, id in enumerate(tracked_ids):
            id_seg = tracked_id_image == id
            expected_segs[b] = id_seg
            missed_offset += 1

        for b, id in enumerate(missed_ids):
            id_seg = missed_id_image == id
            expected_segs[missed_offset + b] = id_seg

        # matched_idx_pair, missed_idx, unmatched_seg_idx = assign_segments(
        #     expected_segs, segs, self.min_iou_thresh
        # )

        Statistics.stopTimer('SegmentTracker_assign_0')
        Statistics.startTimer('SegmentTracker_assign_1')

        half_exp_segs = expected_segs[:, ::2, ::2]
        half_segs = segs[:, ::2, ::2]

        # matched_idx_pair, missed_idx, unmatched_seg_idx = assign_segments(
        #     half_exp_segs, half_segs, self.min_iou_thresh
        # )

        (
            matched_idx_pair,
            missed_idx,
            unmatched_seg_idx,
            c_overlapped_indices,
            r_overlapped_indices,
        ) = assign_segments(half_exp_segs, half_segs, self.min_iou_thresh)

        for r, c in c_overlapped_indices:
            keep = segs[c]
            remove = expected_segs[r]

            inter = (remove & keep).astype(bool)
            segs[c][inter] = 0

        for r, c in r_overlapped_indices:
            keep = expected_segs[r]
            remove = segs[c]

            inter = (remove & keep).astype(bool)
            remove[inter] = 0

        Statistics.stopTimer('SegmentTracker_assign_1')

        Statistics.stopTimer('SegmentTracker_assign')

        Statistics.startTimer('SegmentTracker_results')

        out_tracklet_ids = []
        self.missed_tracklets = {}
        self.tracked_tracklets = {}
        for id_idx, seg_idx in matched_idx_pair:
            id = prev_ids[id_idx]

            seg = segs[seg_idx]
            seg_trl = self.id_tracklet_dict[id]

            seg_trl.add_cam_segment_mask(timestamp, self.cam_index, seg)
            out_tracklet_ids.append(id)
            self.tracked_tracklets[id] = seg_trl

        for id_idx in missed_idx:
            id = prev_ids[id_idx]
            seg_trl = self.id_tracklet_dict[id]
            seg_trl.missed_count += 1

            if seg_trl.missed_count < self.max_missed_track:
                self.missed_tracklets[id] = seg_trl
                seg_trl.predicted_mask = expected_segs[id_idx]

        for seg_idx in unmatched_seg_idx:
            seg = segs[seg_idx]
            seg_trl = SegmentTracklet()
            self.id_tracklet_dict[seg_trl.id] = seg_trl
            seg_trl.add_cam_segment_mask(timestamp, self.cam_index, seg)
            out_tracklet_ids.append(seg_trl.id)

            self.tracked_tracklets[seg_trl.id] = seg_trl

        self.prev_timestamp = timestamp
        self.prev_gray = curr_gray
        Statistics.stopTimer('SegmentTracker_results')

        return out_tracklet_ids

    def _calculate_flow(self, src_gray, dst_gray):
        flow = cv2.calcOpticalFlowFarneback(
            src_gray, dst_gray, None, 0.5, 4, 30, 3, 5, 1.2, 0
        )
        return flow


class SegmentMatcher:
    """
    This class matches the segments detected using the feature matching model.
    """

    def __init__(
        self,
        e2emvm: E2emvm,
        cam_params: Dict[str, any],
        cam_to_trl_ids_dict: Dict[int, Dict[int, List[int]]],
        trl_id_to_trl_dict: Dict[int, SegmentTracklet],
        target_pairs,
        epipolar_thresholds,
        out_debug_dir=None,
    ):
        self._e2emvm = e2emvm
        self._cam_params = list(cam_params.values())
        # _cam_to_trl_ids_dict[timestamp][cam_index] -> list[tracklet_id]
        self._cam_to_trl_ids_dict = cam_to_trl_ids_dict
        # _trl_id_to_trl_dict[id] -> tracklet
        self._trl_id_to_trl_dict = trl_id_to_trl_dict

        # TODO make parameters configurable!
        self._minimum_matched_kpt_count = 4
        self._target_pairs = target_pairs
        self._target_Ec1c0s = {}
        self._epipolar_thresholds = {}

        # prepare essential matrixes and epipolr thresholds
        for i, (idx0, idx1) in enumerate(self._target_pairs):
            key = f'{idx0}_{idx1}'

            T_bc0 = self._cam_params[idx0].T_bc
            T_bc1 = self._cam_params[idx1].T_bc
            R_bc0 = T_bc0[:3, :3]
            P_bc0 = T_bc0[:3, 3]
            R_bc1 = T_bc1[:3, :3]
            P_bc1 = T_bc1[:3, 3]

            R_c1b = R_bc1.T
            P_c1b = -R_c1b @ P_bc1

            R_c1c0 = R_c1b @ R_bc0
            P_c1c0 = R_c1b @ P_bc0 + P_c1b
            x, y, z = P_c1c0
            P_skew = np.array([[0, -z, y], [z, 0, -x], [-y, x, 0]])
            Ec1c0 = P_skew @ R_c1c0
            self._target_Ec1c0s[key] = Ec1c0
            self._epipolar_thresholds[key] = epipolar_thresholds[i]

        # prepare debugs
        self._random_colors = generate_colors()
        self.out_debug_dir = out_debug_dir
        if self.out_debug_dir is not None:
            os.makedirs(self.out_debug_dir, exist_ok=True)
            self._match_video_writers = {}

            for idx0, idx1 in self._target_pairs:
                key = f'{idx0}_{idx1}'
                match_video_path = os.path.join(
                    out_debug_dir, f"match_{key}.mp4"
                )

                self._match_video_writers[key] = cv2.VideoWriter(
                    match_video_path,
                    cv2.VideoWriter_fourcc(*'mp4v'),
                    20,
                    (
                        self._cam_params[0].img_wh[0] * 2,
                        self._cam_params[0].img_wh[1],
                    ),
                )

    def match(
        self,
        timestamp,
        images: List[np.ndarray],
        frame_count: int = -1,
    ):
        """
        :param timestamp: timestamp
        :param images: lits of images from 7 cams
        :param frame_count: frame count for debug

        """

        Statistics.startTimer('SegmentMatcher_extract_features')
        torch_images, features = self._e2emvm.extract_features(images)
        Statistics.stopTimer('SegmentMatcher_extract_features')

        Statistics.startTimer('SegmentMatcher_matching')

        preds = self._e2emvm(torch_images, features, self._target_pairs)
        Statistics.stopTimer('SegmentMatcher_matching')

        all_matches = preds['matches']

        # segment ids when keypoints are projected into id_image
        kpt_seg_ids = []
        undistorted_kpts_list = []

        Statistics.startTimer('SegmentMatcher_undistort')
        # undistort keypoints for epipolar constraints and collect segment ids

        pre_computed_seg_area = {}

        for cam_idx, kpts in enumerate(features['keypoints']):
            kpts = kpts.cpu().numpy().astype(np.float32)
            seg_ids = np.full(kpts.shape[0], -1, dtype=int)

            shape = images[0].shape[:2]
            id_image = np.full(shape, -1.0, dtype=np.float32)

            tracked_trl_ids = self._cam_to_trl_ids_dict[timestamp][cam_idx]
            for id in tracked_trl_ids:
                seg_trl = self._trl_id_to_trl_dict[id]
                seg = seg_trl.query_segment_mask(timestamp, cam_idx)
                id_image[seg > 0] = id
                pre_computed_seg_area[(id, timestamp, cam_idx)] = np.sum(seg)

            for idx, kpt in enumerate(kpts):
                x, y = kpt.astype(int)
                seg_id = id_image[y, x]
                if seg_id >= 0:
                    seg_ids[idx] = seg_id

            kpt_seg_ids.append(seg_ids)

            K = self._cam_params[cam_idx].K
            D = self._cam_params[cam_idx].distort_coefs
            R = np.eye(3, dtype=np.float32)
            P = np.eye(3, dtype=np.float32)
            kpts = kpts.reshape(-1, 1, 2)

            undistorted_kpts = cv2.fisheye.undistortPoints(kpts, K, D, R=R, P=P)
            undistorted_kpts = undistorted_kpts.reshape(-1, 2)
            undistorted_kpts = np.hstack(
                (undistorted_kpts, np.ones((undistorted_kpts.shape[0], 1)))
            )
            undistorted_kpts_list.append(undistorted_kpts)
        Statistics.stopTimer('SegmentMatcher_undistort')

        Statistics.startTimer('SegmentMatcher_assign')

        # find same segment tracklet between camera pairs
        segment_id_pair = {}
        for idx0, idx1 in preds['indices_pairs']:
            key = f'{idx0}_{idx1}'
            segment_id_pair[key] = []

            matches = all_matches[f'matches{idx0}_{key}'][0].cpu().numpy()
            confs = all_matches[f'conf_scores_{key}'][0, :, 0].cpu().numpy()
            seg_ids0 = kpt_seg_ids[idx0]
            seg_ids1 = kpt_seg_ids[idx1]

            valid_indices = np.flatnonzero(
                (matches >= 0) & (confs >= 0.02) & (seg_ids0 >= 0)
            )

            kpts0 = features['keypoints'][idx0].cpu().numpy()
            kpts1 = features['keypoints'][idx1].cpu().numpy()

            kpts0 = kpts0[valid_indices]
            seg_ids0 = seg_ids0[valid_indices]
            undists0 = undistorted_kpts_list[idx0][valid_indices]

            kpts1_idx = matches[valid_indices]
            kpts1 = kpts1[kpts1_idx]
            seg_ids1 = seg_ids1[kpts1_idx]
            undists1 = undistorted_kpts_list[idx1][kpts1_idx]

            valid_indices = np.flatnonzero(seg_ids1 >= 0)

            kpts0 = kpts0[valid_indices]
            seg_ids0 = seg_ids0[valid_indices]
            undists0 = undists0[valid_indices]

            kpts1 = kpts1[valid_indices]
            seg_ids1 = seg_ids1[valid_indices]
            undists1 = undists1[valid_indices]

            E_c1c0 = self._target_Ec1c0s[key]

            undists0 = undists0[:, :, np.newaxis]
            temp = E_c1c0 @ undists0

            undists1 = undists1[:, np.newaxis, :]
            epipolar_constrains = (undists1 @ temp).flatten()

            epipolar_constrains = (
                np.abs(epipolar_constrains) < self._epipolar_thresholds[key]
            )

            before_epi0 = kpts0
            kpts0 = kpts0[epipolar_constrains]
            seg_ids0 = seg_ids0[epipolar_constrains]

            before_epi1 = kpts1
            kpts1 = kpts1[epipolar_constrains]
            seg_ids1 = seg_ids1[epipolar_constrains]

            unique_ids0 = np.unique(seg_ids0.flatten())
            unique_ids1 = np.unique(seg_ids1.flatten())

            id_idx0 = {}
            for idx, id in enumerate(unique_ids0):
                id_idx0[id] = idx

            id_idx1 = {}
            for idx, id in enumerate(unique_ids1):
                id_idx1[id] = idx

            rows = unique_ids0.shape[0]
            cols = unique_ids1.shape[0]

            # construct hit matrix to get best match
            hit_mat = np.zeros((rows, cols))

            for id0, id1 in zip(seg_ids0, seg_ids1):
                r = id_idx0[id0]
                c = id_idx1[id1]
                hit_mat[r, c] += 1

            # reject match when more than two segments are mached with one segment
            for i in range(hit_mat.shape[0]):
                if np.sum(hit_mat[i] > self._minimum_matched_kpt_count) > 1:
                    hit_mat[i] = 0

            for j in range(hit_mat.shape[1]):
                if np.sum(hit_mat[:, j] > self._minimum_matched_kpt_count) > 1:
                    hit_mat[:, j] = 0

            matched_indices = np.array(solve_dense(-hit_mat)).T

            matches = []

            for m in matched_indices:
                if hit_mat[m[0], m[1]] > self._minimum_matched_kpt_count:
                    seg_id0 = unique_ids0[m[0]]
                    seg_id1 = unique_ids1[m[1]]

                    seg0_area = pre_computed_seg_area[
                        (seg_id0, timestamp, idx0)
                    ]
                    seg1_area = pre_computed_seg_area[
                        (seg_id1, timestamp, idx1)
                    ]

                    score = hit_mat[m[0], m[1]] ** 2 / (seg0_area * seg1_area)

                    matches.append(
                        (
                            unique_ids0[m[0]],
                            unique_ids1[m[1]],
                            score,
                        )
                    )

            segment_id_pair[key] = matches

            # save match video
            if self.out_debug_dir is not None:
                self.render_debug_video(
                    timestamp,
                    frame_count,
                    images,
                    idx0,
                    idx1,
                    before_epi0,
                    before_epi1,
                    kpts0,
                    kpts1,
                    matches,
                )

        Statistics.stopTimer('SegmentMatcher_assign')

        return segment_id_pair

    def cleanup(self):
        if self.out_debug_dir is not None:
            for _, writer in self._match_video_writers.items():
                writer.release()

    def render_debug_video(
        self,
        timestamp,
        frame_count,
        images,
        idx0,
        idx1,
        before_epi0,
        before_epi1,
        kpts0,
        kpts1,
        matches,
    ):
        key = f'{idx0}_{idx1}'
        img0 = images[idx0].copy()
        img1 = images[idx1].copy()

        for i, (id0, id1, _) in enumerate(matches):
            seg0 = self._trl_id_to_trl_dict[id0]
            mask0 = seg0.query_segment_mask(timestamp, idx0)
            draw_mask(img0, mask0, i, 0.8)
            seg1 = self._trl_id_to_trl_dict[id1]
            mask1 = seg1.query_segment_mask(timestamp, idx1)
            draw_mask(img1, mask1, i, 0.8)

        stitched_img = np.concatenate([img0, img1], axis=1)
        vis_img = stitched_img.copy()

        vis_lines_before = np.concatenate(
            [before_epi0, before_epi1], axis=1
        ).astype(np.int32)
        vis_lines_before[:, 2] += img0.shape[1]

        vis_lines = np.concatenate([kpts0, kpts1], axis=1).astype(np.int32)
        vis_lines[:, 2] += img0.shape[1]

        for i, line in enumerate(vis_lines_before):
            c = (0, 0, 0)
            line = line.tolist()
            pt1, pt2 = (line[0], line[1]), (line[2], line[3])
            cv2.line(vis_img, pt1, pt2, color=c, lineType=cv2.LINE_AA)
            cv2.circle(vis_img, pt1, radius=4, color=c, lineType=cv2.LINE_AA)
            cv2.circle(vis_img, pt2, radius=4, color=c, lineType=cv2.LINE_AA)

        for i, line in enumerate(vis_lines):
            c = self._random_colors[i % len(self._random_colors)]
            line = line.tolist()
            pt1, pt2 = (line[0], line[1]), (line[2], line[3])
            cv2.line(vis_img, pt1, pt2, color=c, lineType=cv2.LINE_AA)
            cv2.circle(vis_img, pt1, radius=4, color=c, lineType=cv2.LINE_AA)
            cv2.circle(vis_img, pt2, radius=4, color=c, lineType=cv2.LINE_AA)
        cv2.putText(
            vis_img,
            f"{frame_count}",
            (40, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 255),
            1,
            cv2.LINE_AA,
        )

        self._match_video_writers[key].write(vis_img)


class CameraTrackingHandler(DbHandler):
    def __init__(
        self,
        npy_data,
        obj_seg_model: SegmentationYolo,
        e2emvm: E2emvm,
        cache_dir,
        ego_mask_dir,
        out_debug_dir=None,
    ):
        """Handler to run model inference and tracking for all cameras."""
        self.npy_data = npy_data
        self._db_name = npy_data['db_filename']
        self._vehicle_name = npy_data['vehicle_info']['tname']
        self._cache_dir = cache_dir
        self._frame_count = 0

        # Camera initialization
        sensors = npy_data['sensors']
        self._cam_params = get_all_cameras_params(sensors)

        # outputs
        # _cam_to_trl_ids_dict[timestamp][cam_index] -> List[tracklet_id]
        self._cam_to_trl_ids_dict: Dict[int, Dict[int, List[int]]] = {}
        # _trl_id_to_trl_dict[tracklet_id]->tracklet
        self._trl_id_to_trl_dict: Dict[int, SegmentTracklet] = {}

        # Prepare segment tracking
        self._seg_model = obj_seg_model
        self._ego_masks = load_ego_masks(ego_mask_dir, self._vehicle_name)
        self._segment_trackers = [
            SegmentTracker(cam_idx, self._trl_id_to_trl_dict)
            for cam_idx in range(len(self._cam_params))
        ]

        # Prepare segment matching
        # TODO make parameters configurable!
        self._target_pairs = [
            # front
            (0, 1),
            (0, 2),
            # left
            (2, 3),
            (3, 5),
            # right
            (2, 4),
            (4, 6),
        ]
        _epipolar_thresholds = [
            # front    # front
            0.1,  #    # (0, 1),
            0.001,  #    # (0, 2),
            # left     # left
            0.1,  #    # (2, 3),
            0.07,  #    # (3, 5),
            # right    # right
            0.1,  #    # (2, 4),
            0.1,  #    # (4, 6),
        ]

        self._segment_matcher = SegmentMatcher(
            e2emvm,
            self._cam_params,
            self._cam_to_trl_ids_dict,
            self._trl_id_to_trl_dict,
            self._target_pairs,
            _epipolar_thresholds,
            out_debug_dir,
        )

        # _seg_id_pairs_dict[idx0_idx1]-> List[matched_id_pairs]
        self._seg_id_pairs_dict = {}
        for idx0, idx1 in self._target_pairs:
            key = f'{idx0}_{idx1}'
            self._seg_id_pairs_dict[key] = []

        # debug option for intermideate results
        self.out_debug_dir = out_debug_dir

        if self.out_debug_dir is not None:
            video_width = -1
            video_height = -1

            self._video_pos_s = []
            for _, cam_param in self._cam_params.items():
                w, h = cam_param.video_pos + cam_param.img_wh
                video_width = max(video_width, w)
                video_height = max(video_height, h)
                self._video_pos_s.append(cam_param.video_pos)

            seg_video_path = os.path.join(self.out_debug_dir, f'segs.mp4')
            self._seg_video_writer = VideoWriter(
                seg_video_path, video_width, video_height, fps=20
            )

            trl_before_matching_video_path = os.path.join(
                self.out_debug_dir, f'before_matching.mp4'
            )
            self._trl_before_matching_video_writer = VideoWriter(
                trl_before_matching_video_path,
                video_width,
                video_height,
                fps=20,
            )

            trl_after_matching_video_dir = os.path.join(
                self.out_debug_dir, f"after_matching_temp.mp4"
            )
            self._trl_after_matching_video_writer = VideoWriter(
                trl_after_matching_video_dir,
                video_width,
                video_height,
                fps=20,
            )

    def _handle_cam_msg(self, timestamp, seven_cam_image):
        timestamp = int(timestamp * 1e9)

        imgs = []

        for _, cam_param in self._cam_params.items():
            img = extract_image(
                seven_cam_image, cam_param.img_wh, cam_param.video_pos
            )
            imgs.append(img)

        seg_results = self._seg_model.segment_batch(imgs)

        self._cam_to_trl_ids_dict[timestamp] = {}
        cam_names = list(self._cam_params.keys())
        for idx, seg_result in enumerate(seg_results):
            seg_result['masks'] = remove_ego_body_from_masks(
                seg_result['masks'], self._ego_masks[cam_names[idx]]
            )
            seg_result['masks'] = remove_overlapped_area(
                seg_result['masks'], seg_result['scores']
            )

            curr_tracklet_ids = self._segment_trackers[idx].track(
                timestamp, imgs[idx], seg_result['masks']
            )

            self._cam_to_trl_ids_dict[timestamp][idx] = curr_tracklet_ids

        # match segments between cams
        new_id_pairs = self._segment_matcher.match(
            timestamp,
            imgs,
            self._frame_count,  # for debugging
        )

        for idx0, idx1 in self._target_pairs:
            key = f'{idx0}_{idx1}'
            self._seg_id_pairs_dict[key].append(new_id_pairs[key])

        if self.out_debug_dir:
            self._render_debug_videos(timestamp, imgs, seg_results)

        # to_save_path = './cache/debug'
        # for idx in range(len(imgs)):
        #     save_idx_path = os.path.join(to_save_path, f'{idx}')
        #     os.makedirs(save_idx_path, exist_ok=True)
        #     save_image_path = os.path.join(
        #         save_idx_path, f'{self._frame_count}.png'
        #     )
        #     cv2.imwrite(save_image_path, imgs[idx])

    def get_topics(self):
        return [DbTopic.CAM_MSG]

    def __call__(self, timestamp, topic, data):
        try:
            if topic == DbTopic.CAM_MSG:
                self._handle_cam_msg(timestamp, data)
                self._frame_count += 1
                # Statistics.reportAll()
        except KeyboardInterrupt:
            raise KeyboardInterrupt
        except Exception as e:
            logging.error(
                f'Exception at {timestamp}, Topic: {topic}, Db: {self._db_name}'
                f'\n{traceback.format_exc()}'
            )
            raise e

    def get_processed_data(self):
        """
        Before returning processed data, this functions handles all previous correspondence matches
        to remove conflict between cameras.
        Firstly, it finds conflictc in each target pairs and remove false-positives.
        Finally, it groups results to generate a unified tracklet
        """
        groups = []

        for idx0, idx1 in self._target_pairs:
            pairs_list = self._seg_id_pairs_dict[f'{idx0}_{idx1}']

            graph01 = defaultdict(lambda: defaultdict(lambda: 0.0))
            graph10 = defaultdict(lambda: defaultdict(lambda: 0.0))

            for pairs in pairs_list:
                if len(pairs) < 1:
                    continue
                for pair in pairs:
                    id0, id1, score = pair
                    curr_score01 = graph01[id0][id1]

                    if score > curr_score01:
                        graph01[id0][id1] = score
                        graph10[id1][id0] = score

            to_remove_edge = []
            id1_pairs = []
            for id0, id1_to_score in graph01.items():
                if len(id1_to_score) < 2:
                    continue

                id1s = list(id1_to_score.keys())
                id1_pairs = list(itertools.combinations(id1s, 2))
                for id1_0, id1_1 in id1_pairs:
                    trl1_0 = self._trl_id_to_trl_dict[id1_0]
                    timestamps0 = set(trl1_0.cam_to_segment_mask.keys())

                    trl1_1 = self._trl_id_to_trl_dict[id1_1]
                    timestamps1 = set(trl1_1.cam_to_segment_mask.keys())

                    conflict = bool(timestamps0 & timestamps1)

                    if not conflict:
                        continue

                    score0_1_0 = graph01[id0][id1_0]
                    score0_1_1 = graph01[id0][id1_1]

                    if score0_1_0 > score0_1_1:
                        to_remove_edge.append((id0, id1_1))
                    else:
                        to_remove_edge.append((id0, id1_0))

            for id0, id1 in to_remove_edge:
                if id0 in graph01:
                    if id1 in graph01[id0]:
                        del graph01[id0][id1]

                    if len(graph01[id0]) == 0:
                        del graph01[id0]
                if id1 in graph10:
                    if id0 in graph10[id1]:
                        del graph10[id1][id0]

                    if len(graph10[id1]) == 0:
                        del graph10[id1]

            to_remove_edge = []
            id0_pairs = []
            for id1, id0_to_score in graph10.items():
                if len(id0_to_score) < 2:
                    continue

                id0s = list(id0_to_score.keys())
                id0_pairs = list(itertools.combinations(id0s, 2))

                for id0_0, id0_1 in id0_pairs:
                    trl0_1 = self._trl_id_to_trl_dict[id0_0]
                    timestamps0 = set(trl0_1.cam_to_segment_mask.keys())

                    trl0_1 = self._trl_id_to_trl_dict[id0_1]
                    timestamps1 = set(trl0_1.cam_to_segment_mask.keys())

                    conflict = bool(timestamps0 & timestamps1)

                    if not conflict:
                        continue

                    score1_0_0 = graph01[id1][id0_0]
                    score1_0_1 = graph01[id1][id0_1]

                    if score1_0_0 > score1_0_1:
                        to_remove_edge.append((id0_1, id1))
                    else:
                        to_remove_edge.append((id0_1, id1))

            for id0, id1 in to_remove_edge:
                if id0 in graph01:
                    if id1 in graph01[id0]:
                        del graph01[id0][id1]

                    if len(graph01[id0]) == 0:
                        del graph01[id0]
                if id1 in graph10:
                    if id0 in graph10[id1]:
                        del graph10[id1][id0]

                    if len(graph10[id1]) == 0:
                        del graph10[id1]

            graph01.update(graph10)

            visited = set()
            for node in graph01:
                if node not in visited:
                    components = set()
                    dfs(graph01, node, visited, components)
                    groups.append(components)

        merged = []
        while groups:
            first = groups.pop(0)
            merged_group = first
            i = 0
            while i < len(groups):
                if merged_group & groups[i]:
                    merged_group |= groups.pop(i)
                    i = 0
                else:
                    i += 1

            merged.append(merged_group)

        for group in merged:
            to_merge_trl = []
            for id in group:
                to_merge_trl.append(self._trl_id_to_trl_dict[id])
            merged_trl = SegmentTracklet()

            for trl in to_merge_trl:

                for (
                    timestamp,
                    cam_idx_to_masks,
                ) in trl.cam_to_segment_mask.items():
                    merged_trl.cam_to_segment_mask.setdefault(
                        timestamp, {}
                    ).update(cam_idx_to_masks)

                del self._trl_id_to_trl_dict[trl.id]

                for (
                    timestamp,
                    cam_idx_to_masks,
                ) in trl.cam_to_segment_mask.items():
                    for cam_idx, _ in cam_idx_to_masks.items():
                        ids = self._cam_to_trl_ids_dict[timestamp][cam_idx]
                        ids.remove(trl.id)

                        if merged_trl.id not in ids:
                            ids.append(merged_trl.id)

            self._trl_id_to_trl_dict[merged_trl.id] = merged_trl
            merged_trl.merged = True

        if self.out_debug_dir is not None:
            self._render_merged_debug_videos()

        return {
            'cam_to_trl_ids_dict': self._cam_to_trl_ids_dict,
            'trl_id_to_trl_dict': self._trl_id_to_trl_dict,
        }

    def cleanup(self):

        self._segment_matcher.cleanup()

        if self.out_debug_dir is not None:
            self._trl_before_matching_video_writer.release()
            self._seg_video_writer.release()

    def _render_debug_videos(self, timestamp, imgs, seg_results):
        # record extracted segments
        vis_imgs = []
        for img, seg_result in zip(imgs, seg_results):
            segs = seg_result['masks']
            img = img.copy()
            for idx, seg in enumerate(segs):
                draw_mask(img, seg, idx, 0.8)
            vis_imgs.append(img)

        video_frame = collate_images(vis_imgs, self._video_pos_s)
        cv2.putText(
            video_frame,
            f'{timestamp:.6f} - {self._frame_count}',
            (40, 60),
            cv2.FONT_HERSHEY_SIMPLEX,
            2,
            (0, 255, 255),
            3,
            cv2.LINE_AA,
        )
        self._seg_video_writer.write(video_frame)

        # record tracklets before correspondence matching
        vis_imgs = []
        for idx, trl_ids in self._cam_to_trl_ids_dict[timestamp].items():
            img = imgs[idx].copy()
            for id in trl_ids:
                trl = self._trl_id_to_trl_dict[id]
                mask = trl.query_segment_mask(
                    timestamp,
                    idx,
                )
                draw_mask(img, mask, id, 0.8)
            vis_imgs.append(img)

        video_frame = collate_images(vis_imgs, self._video_pos_s)
        cv2.putText(
            video_frame,
            f'{timestamp} - {self._frame_count}',
            (40, 60),
            cv2.FONT_HERSHEY_SIMPLEX,
            2,
            (0, 255, 255),
            3,
            cv2.LINE_AA,
        )
        self._trl_before_matching_video_writer.write(video_frame)

        # record origin images for later correspondence matching video
        vis_imgs = []
        for image in imgs:
            image = image.copy()
            vis_imgs.append(image)

        video_frame = collate_images(vis_imgs, self._video_pos_s)
        cv2.putText(
            video_frame,
            f"{timestamp} - {self._frame_count}",
            (40, 60),
            cv2.FONT_HERSHEY_SIMPLEX,
            2,
            (0, 255, 255),
            3,
            cv2.LINE_AA,
        )
        self._trl_after_matching_video_writer.write(video_frame)

    def _render_merged_debug_videos(self):
        self._trl_after_matching_video_writer.release()
        temp_trl_after_matching_video_dir = os.path.join(
            self.out_debug_dir, f"after_matching_temp.mp4"
        )

        cap = cv2.VideoCapture(temp_trl_after_matching_video_dir)
        video_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        video_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        trl_after_matching_video_dir = os.path.join(
            self.out_debug_dir, f"after_matching.mp4"
        )
        self._trl_after_matching_video_writer = VideoWriter(
            trl_after_matching_video_dir, video_width, video_height, fps=20
        )

        cam_params = list(self._cam_params.values())
        # record tracklets after correspondence matching
        for frame_idx, (timestamp, cam_idx_to_trl_ids) in enumerate(
            self._cam_to_trl_ids_dict.items()
        ):
            ret, seven_cam_image = cap.read()
            if not ret:
                raise SystemError("saved video is missing")

            frames = []
            for cam_idx, trl_ids in cam_idx_to_trl_ids.items():
                cam_param = cam_params[cam_idx]
                image = extract_image(
                    seven_cam_image, cam_param.img_wh, cam_param.video_pos
                ).copy()

                for trl_id in trl_ids:
                    trl = self._trl_id_to_trl_dict[trl_id]
                    mask = trl.query_segment_mask(timestamp, cam_idx)
                    draw_mask(image, mask, trl_id, 0.8)

                frames.append(image)

            video_frame = collate_images(frames, self._video_pos_s)

            self._trl_after_matching_video_writer.write(video_frame)

        self._trl_after_matching_video_writer.release()
        os.remove(temp_trl_after_matching_video_dir)


import gc

for var in dir():
    if isinstance(globals()[var], torch.Tensor):
        del globals()[var]

torch.cuda.empty_cache()
gc.collect()

out_debug_dir = None

if configs['debug']:
    out_debug_dir = os.path.join(out_dir_path, npy_name)
    os.makedirs(out_debug_dir, exist_ok=True)

camera_tracking_handler = CameraTrackingHandler(
    npy_data=npy_data,
    obj_seg_model=obj_seg_model,
    e2emvm=e2emvm,
    cache_dir=configs['diskcache_dir'],
    ego_mask_dir=configs['ego_mask_dir'],
    out_debug_dir=out_debug_dir,
)

run_segmented_databases(db_paths, db_s_time, duration, camera_tracking_handler)

camera_tracking_handler.get_processed_data()
camera_tracking_handler.cleanup()

Loading /media/vol/shared/obstacle/models/yolosegment/best.torchscript for TorchScript inference...


In [3]:
trl_id_to_trl_dict = camera_tracking_handler._trl_id_to_trl_dict
trl_id_to_trl_dict[3].cam_to_segment_mask.keys()

dict_keys([3450748466600, 3450802483584, 3450859682415, 3450904322375, 3450959153861, 3451000175631, 3451059549319, 3451107781802])

In [4]:
class CameraImageHandler(DbHandler):
    def __init__(
        self,
        npy_data,
        cache_dir,
        out_debug_dir=None,
    ):
        """Handler to run model inference and tracking for all cameras."""
        self.npy_data = npy_data
        self._db_name = npy_data['db_filename']
        self._vehicle_name = npy_data['vehicle_info']['tname']
        self._cache_dir = cache_dir
        self._frame_count = 0

        # Camera initialization
        sensors = npy_data['sensors']
        self._cam_params = get_all_cameras_params(sensors)
        self._cam_param_list = list(self._cam_params.values())

        self.time_to_images = {}
        # debug option for intermideate results
        self.out_debug_dir = out_debug_dir

        if self.out_debug_dir is not None:
            video_width = -1
            video_height = -1

            self._video_pos_s = []
            for _, cam_param in self._cam_params.items():
                w, h = cam_param.video_pos + cam_param.img_wh
                video_width = max(video_width, w)
                video_height = max(video_height, h)
                self._video_pos_s.append(cam_param.video_pos)

            origin_video_path = os.path.join(self.out_debug_dir, f'origin.mp4')
            self._origin_video_writer = VideoWriter(
                origin_video_path, video_width, video_height, fps=20
            )

    def _handle_cam_msg(self, timestamp, seven_cam_image):
        timestamp = int(timestamp * 1e9)

        imgs = []

        for _, cam_param in self._cam_params.items():
            img = extract_image(
                seven_cam_image, cam_param.img_wh, cam_param.video_pos
            )
            imgs.append(img)
        self.time_to_images[timestamp] = imgs

        if self.out_debug_dir:
            frame = collate_images(imgs, self._video_pos_s)
            self._origin_video_writer.write(frame)

    def get_topics(self):
        return [DbTopic.CAM_MSG]

    def __call__(self, timestamp, topic, data):
        try:
            if topic == DbTopic.CAM_MSG:
                self._handle_cam_msg(timestamp, data)
                self._frame_count += 1

        except KeyboardInterrupt:
            raise KeyboardInterrupt
        except Exception as e:
            logging.error(
                f'Exception at {timestamp}, Topic: {topic}, Db: {self._db_name}'
                f'\n{traceback.format_exc()}'
            )
            raise e

    def cleanup(self):

        if self.out_debug_dir is not None:
            self._origin_video_writer.release()
            ts = list(self.time_to_images.keys())

            to_save = np.array(ts)
            np.save('saves_ns.npy', to_save)


camera_image_handler = CameraImageHandler(
    npy_data=npy_data,
    cache_dir=configs['diskcache_dir'],
    out_debug_dir=out_debug_dir,
)

run_segmented_databases(db_paths, db_s_time, duration, camera_image_handler)

camera_image_handler.cleanup()

In [5]:
trl_id_to_trl_dict.keys()

dict_keys([3, 4, 5, 6, 7, 8, 12, 13, 14, 15, 16, 17, 18, 19, 20, 22, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73])

In [6]:
time_to_imgs = camera_image_handler.time_to_images

for trl_id in reversed(list(trl_id_to_trl_dict.keys())):
    trl_id = 241
    for time, idx_to_mask in trl_id_to_trl_dict[
        trl_id
    ].cam_to_segment_mask.items():
        for idx, mask in idx_to_mask.items():
            print(f'trl_id : {trl_id} sum : {np.sum(mask)}')

            temp_image = time_to_imgs[time][idx]
            debug = temp_image.copy()
            draw_mask(debug, mask, 0, 1.0)

            cv2.imshow("", debug)
            key = cv2.waitKey(0)
            if key == 27:
                break
            # break

        if key == 27:
            break
        # break
    if key == 27:
        break

cv2.destroyAllWindows()

KeyError: 241

In [ ]:
trl_id = 241


cam_params = camera_image_handler._cam_param_list
cam_to_mask = trl_id_to_trl_dict[trl_id].cam_to_segment_mask
total = 0

h, w = (0, 0)
for time, idx_to_mask in cam_to_mask.items():
    for idx, mask in idx_to_mask.items():
        h, w = mask.shape
        total += 1

# timestamp, idx, K, D, image, mask
type = np.dtype(
    [
        ('timestamp', int),
        ('idx', int),
        ('K', float, (3, 3)),
        ('D', float, (4,)),
        ('image', np.uint8, (h, w, 3)),
        ('mask', np.uint8, (h, w)),
    ]
)


data = np.empty(total, dtype=type)

count = 0
for time, idx_to_mask in cam_to_mask.items():
    for idx, mask in idx_to_mask.items():
        K = cam_params[idx].K
        D = cam_params[idx].distort_coefs
        data[count] = (time, idx, K, D, time_to_imgs[time][idx], mask)
        count += 1

np.save('tracklet.npy', data)

# for time, idx_to_mask in trl_id_to_trl_dict[trl_id].cam_to_segment_mask.items():
#     for idx, mask in idx_to_mask.items():
#         print(f'trl_id : {trl_id} sum : {np.sum(mask)}')

#         temp_image = time_to_imgs[time][idx]
#         debug = temp_image.copy()
#         draw_mask(debug, mask, 0, 1.0)

#         cv2.imshow("", debug)
#         key = cv2.waitKey(0)
#         if key == 27:
#             break
#         # break

#     if key == 27:
#         break

In [ ]:
tracklet_npy = np.load('tracklet.npy')

print(tracklet_npy.dtype)

for i in range(tracklet_npy.shape[0]):
    ns = tracklet_npy['timestamp'][i]
    idx = tracklet_npy['idx'][i]
    K = tracklet_npy['K'][i]
    D = tracklet_npy['D'][i]
    img = tracklet_npy['image'][i]
    mask = tracklet_npy['mask'][i]

    print(np.sum(mask))

    render = img.copy()
    draw_mask(render, mask)
    cv2.imshow("", render)
    key = cv2.waitKey(0)
    if key == 27:
        break

cv2.destroyAllWindows()

[('timestamp', '<i8'), ('idx', '<i8'), ('K', '<f8', (3, 3)), ('D', '<f8', (4,)), ('image', 'u1', (468, 832, 3)), ('mask', 'u1', (468, 832))]
3331
105826
58923
5143
100069
27238
7034
92519
13998
8634
84674
9912
76112
11307
67950
12778
59989
14043
51900
15381
45535
